# Support Ticket Classifier · buyer quickstart

Run Spanish, Portuguese, and English ticket triage inside your AWS account. This notebook includes its own synthetic examples, real-time inference, batch transform, and cleanup. No external dataset is needed.

**Release candidate:** use only the package ARN from the approved AWS Marketplace subscription. This source notebook is prepared before publication; a public, approved package is not yet available.

You need a subscription, a SageMaker execution role with access to this product and the chosen S3 bucket, and permission to create SageMaker resources and pass that role. Set the region to one supported by your subscription. Software and AWS infrastructure charges apply. Endpoint resources are deleted in `finally`; batch jobs incur charges until they finish. S3 output remains until the final cleanup cell is run.


In [ ]:
%pip install -q "boto3>=1.34,<2"

In [ ]:
import json
import os
import re
import time
import uuid

import boto3
from botocore.config import Config

REGION = os.environ.get("AWS_REGION", "us-east-1")
MODEL_PACKAGE_ARN = os.environ.get("MODEL_PACKAGE_ARN", "PASTE_SUBSCRIBED_MODEL_PACKAGE_ARN")
EXECUTION_ROLE_ARN = os.environ.get("SAGEMAKER_ROLE_ARN", "PASTE_SAGEMAKER_EXECUTION_ROLE_ARN")
S3_BUCKET = os.environ.get("SAGEMAKER_BUCKET", "PASTE_EXISTING_PRIVATE_BUCKET_NAME")
INSTANCE_TYPE = "ml.c5.xlarge"

if not MODEL_PACKAGE_ARN.startswith("arn:aws:sagemaker:") or ":model-package/" not in MODEL_PACKAGE_ARN:
    raise ValueError("Set MODEL_PACKAGE_ARN from your subscribed product in this region.")
if not EXECUTION_ROLE_ARN.startswith("arn:aws:iam:") or ":role/" not in EXECUTION_ROLE_ARN:
    raise ValueError("Set EXECUTION_ROLE_ARN to your SageMaker execution role.")
if S3_BUCKET.startswith("PASTE_"):
    raise ValueError("Set S3_BUCKET to an existing private bucket in REGION.")

session = boto3.Session(region_name=REGION)
sm = session.client("sagemaker")
runtime = session.client("sagemaker-runtime", config=Config(
    read_timeout=70, connect_timeout=10, retries={"max_attempts": 0}))
s3 = session.client("s3")
run_name = os.environ.get("SAGEMAKER_RUN_NAME", "ss-ticket-" + uuid.uuid4().hex[:12])
if not re.fullmatch(r"[A-Za-z0-9](?:[A-Za-z0-9-]*[A-Za-z0-9])?", run_name) or len(run_name) > 55:
    raise ValueError("Use a unique resource name with at most 55 letters/digits/hyphens.")
prefix = os.environ.get("SAGEMAKER_S3_PREFIX", "software-sushi/examples/" + run_name).strip("/")
if not prefix or ".." in prefix.split("/"):
    raise ValueError("Use a non-empty, dedicated S3 prefix.")
if MODEL_PACKAGE_ARN.split(":")[3] != REGION:
    raise ValueError("MODEL_PACKAGE_ARN and REGION must match.")
expected_revision = os.environ.get("EXPECTED_MODEL_REVISION", "")
resource_tags = [{"Key": "SoftwareSushiExample", "Value": run_name}]


def check_prediction(prediction, ticket):
    if prediction.get("id") != ticket["id"] or prediction.get("model") != "support-ticket-classifier":
        raise ValueError("Unexpected model identity or ticket ID in response.")
    if expected_revision and prediction.get("model_revision") != expected_revision:
        raise ValueError("The endpoint does not match the expected model revision.")
    language = {"DEMO-ES": "es", "DEMO-PT": "pt", "DEMO-EN": "en"}[ticket["id"]]
    if prediction["language"]["label"] != language:
        raise ValueError("Unexpected language for an included synthetic example.")


samples = [
    {
        "id": "DEMO-ES",
        "subject": "Cobro duplicado",
        "body": "Me cobraron dos veces la suscripción. Necesito revisar la factura hoy.",
    },
    {
        "id": "DEMO-PT",
        "subject": "Acesso bloqueado",
        "body": "Não consigo entrar na minha conta. A senha não funciona e preciso de ajuda urgente.",
    },
    {
        "id": "DEMO-EN",
        "subject": "Annual business plan",
        "body": "Could you send a quote for an annual plan for twenty users? We are happy with the service.",
    },
]

## Real-time inference

A ticket can contain `text`, or `subject` and `body`. The response preserves `id` and includes `category`, `urgency`, `sentiment`, `language`, and `top_k`. Model scores are uncalibrated. Route `needs_review` results to a person; review also applies to unknown languages, truncation and experimental custom labels.

The following cell provisions one endpoint, invokes the included samples, and deletes the endpoint, configuration and model on success or failure. Provisioning can take several minutes. Do not stop the kernel before cleanup; if interrupted, use the resource names printed below to clean up in the SageMaker console.


In [ ]:
model_created = config_created = endpoint_created = False
print({"model": run_name, "endpoint_config": run_name, "endpoint": run_name})
try:
    sm.create_model(
        ModelName=run_name,
        ExecutionRoleArn=EXECUTION_ROLE_ARN,
        EnableNetworkIsolation=True,
        PrimaryContainer={"ModelPackageName": MODEL_PACKAGE_ARN},
        Tags=resource_tags,
    )
    model_created = True
    sm.create_endpoint_config(
        EndpointConfigName=run_name,
        ProductionVariants=[
            {
                "VariantName": "AllTraffic",
                "ModelName": run_name,
                "InitialInstanceCount": 1,
                "InstanceType": INSTANCE_TYPE,
                "InitialVariantWeight": 1.0,
            }
        ],
        Tags=resource_tags,
    )
    config_created = True
    sm.create_endpoint(EndpointName=run_name, EndpointConfigName=run_name, Tags=resource_tags)
    endpoint_created = True
    sm.get_waiter("endpoint_in_service").wait(EndpointName=run_name, WaiterConfig={"Delay": 30, "MaxAttempts": 60})
    realtime_outputs = []
    for ticket in samples:
        response = runtime.invoke_endpoint(
            EndpointName=run_name,
            ContentType="application/json",
            Accept="application/json",
            Body=json.dumps(ticket, ensure_ascii=False).encode("utf-8"),
        )
        try:
            prediction = json.loads(response["Body"].read())
        finally:
            response["Body"].close()
        check_prediction(prediction, ticket)
        realtime_outputs.append(prediction)
    print(json.dumps(realtime_outputs, ensure_ascii=False, indent=2))
finally:
    if endpoint_created:
        sm.delete_endpoint(EndpointName=run_name)
        sm.get_waiter("endpoint_deleted").wait(EndpointName=run_name, WaiterConfig={"Delay": 20, "MaxAttempts": 90})
    if config_created:
        sm.delete_endpoint_config(EndpointConfigName=run_name)
    if model_created:
        sm.delete_model(ModelName=run_name)

## Batch transform

The examples are uploaded as UTF-8 JSON Lines. `SingleRecord` and `SplitType=Line` send one ticket per invocation. The container runs without network access; SageMaker handles S3 input and output.

This cell waits for completion before deleting the temporary model. If the notebook is interrupted, inspect the printed job name in SageMaker. A model cannot be safely cleaned up until its job has stopped. Batch output is retained under the unique S3 prefix printed below.


In [ ]:
batch_name = run_name + "-batch"
input_key = prefix + "/input/tickets.jsonl"
output_prefix = prefix + "/output/"
s3.put_object(
    Bucket=S3_BUCKET,
    Key=input_key,
    Body=("\n".join(json.dumps(ticket, ensure_ascii=False) for ticket in samples) + "\n").encode("utf-8"),
    ContentType="application/jsonlines",
)
print({"job": batch_name, "output": f"s3://{S3_BUCKET}/{output_prefix}"})
batch_model_created = job_started = job_terminal = False
try:
    sm.create_model(
        ModelName=batch_name,
        ExecutionRoleArn=EXECUTION_ROLE_ARN,
        EnableNetworkIsolation=True,
        PrimaryContainer={"ModelPackageName": MODEL_PACKAGE_ARN},
        Tags=resource_tags,
    )
    batch_model_created = True
    sm.create_transform_job(
        TransformJobName=batch_name,
        ModelName=batch_name,
        BatchStrategy="SingleRecord",
        MaxConcurrentTransforms=1,
        MaxPayloadInMB=6,
        TransformInput={
            "DataSource": {"S3DataSource": {"S3DataType": "S3Prefix", "S3Uri": f"s3://{S3_BUCKET}/{input_key}"}},
            "ContentType": "application/jsonlines",
            "SplitType": "Line",
        },
        TransformOutput={
            "S3OutputPath": f"s3://{S3_BUCKET}/{output_prefix}",
            "Accept": "application/jsonlines",
            "AssembleWith": "Line",
        },
        TransformResources={"InstanceType": INSTANCE_TYPE, "InstanceCount": 1},
        Tags=resource_tags,
    )
    job_started = True
    deadline = time.monotonic() + 1800
    while time.monotonic() < deadline:
        job = sm.describe_transform_job(TransformJobName=batch_name)
        if job["TransformJobStatus"] in {"Completed", "Failed", "Stopped"}:
            job_terminal = True
            break
        time.sleep(20)
    if not job_terminal:
        raise TimeoutError("Batch exceeded 30 minutes; requesting stop during cleanup.")
    if job["TransformJobStatus"] != "Completed":
        raise RuntimeError(job.get("FailureReason", job["TransformJobStatus"]))
    result = s3.get_object(Bucket=S3_BUCKET, Key=output_prefix + "tickets.jsonl.out")
    try:
        batch_outputs = [
            json.loads(line) for line in result["Body"].read().decode("utf-8").splitlines() if line.strip()
        ]
    finally:
        result["Body"].close()
    if len(batch_outputs) != len(samples):
        raise ValueError("Unexpected batch output count.")
    for prediction, ticket in zip(batch_outputs, samples, strict=True):
        check_prediction(prediction, ticket)
    print(json.dumps(batch_outputs, ensure_ascii=False, indent=2))
finally:
    if job_started and not job_terminal:
        current = sm.describe_transform_job(TransformJobName=batch_name)
        if current["TransformJobStatus"] not in {"Completed", "Failed", "Stopped", "Stopping"}:
            sm.stop_transform_job(TransformJobName=batch_name)
        stop_deadline = time.monotonic() + 300
        while time.monotonic() < stop_deadline:
            current = sm.describe_transform_job(TransformJobName=batch_name)
            if current["TransformJobStatus"] in {"Completed", "Failed", "Stopped"}:
                job_terminal = True
                break
            time.sleep(10)
        if not job_terminal:
            print("Confirm job stop before deleting model:", batch_name)
    if batch_model_created and (not job_started or job_terminal):
        sm.delete_model(ModelName=batch_name)

## Remove this notebook run's S3 objects

Run after checking the outputs. This only deletes the known input and result keys for this unique run; it does not delete the bucket or other objects. SageMaker retains job history. Versioned buckets may retain older object versions.


In [ ]:
deleted = s3.delete_objects(
    Bucket=S3_BUCKET, Delete={"Objects": [{"Key": input_key}, {"Key": output_prefix + "tickets.jsonl.out"}]}
)
if deleted.get("Errors"):
    raise RuntimeError("Some example object deletions failed; inspect the response codes.")
print("Removed the input and output keys for", run_name)

## Operational limits and evaluation

- ES/PT/EN only; unknown and mixed-language messages require review.
- First 128 tokenizer tokens, at most 32 records per HTTP request and 6 MiB total. Blank text is rejected. Limit ticket text to 100,000 characters.
- `429` means the two inference slots are busy; retry with exponential backoff and jitter. Invalid input is `400`, unsupported media `415`.
- Default categories: billing, technical_issue, account_access, cancellation, shipping_delivery, refund, complaint, sales_inquiry, other.
- Custom labels use an experimental zero-shot path. Published quality targets apply to the default taxonomy only. Scores are not calibrated probabilities; `needs_review` is a heuristic, not an accuracy guarantee.
- The pilot evaluation is synthetic and small. Validate with representative, appropriately governed tickets from your own workflow before automating routing. This model does not redact personal data.

Support: info@softwaresushi.com. Include the model version/revision and request ID, without ticket contents or credentials.
